# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [1]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [2]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['CMUATDGUYK', 'OOKSOZBOOP'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 3, 13, 21,  1, 20,  4,  7, 21, 25, 11],
       [15, 15, 11, 19, 15, 26,  2, 15, 15, 16]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0, 11, 25, 21,  7,  4, 20,  1, 21, 13],
       [ 0, 16, 15, 15,  2, 26, 15, 19, 11, 15]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[11, 25, 21,  7,  4, 20,  1, 21, 13,  3],
       [16, 15, 15,  2, 26, 15, 19, 11, 15, 15]], dtype=int32)>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [3]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.hidden = 128
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        # 不需要定义 self.decoder = RNN(...)，因为我们要在 call 里手动写循环
        
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
    def call(self, enc_ids, dec_ids):
        enc_out, state_list = self.encode(enc_ids)
        state = state_list[0]
        dec_emb = self.embed_layer(dec_ids)
        batch_sz = tf.shape(dec_ids)[0]
        seq_len = tf.shape(dec_ids)[1]
        
        all_logits = []
        enc_out_transformed = self.dense_attn(enc_out)
        
        for t in range(seq_len):
            step_input = dec_emb[:, t, :]
            _, next_state = self.decoder_cell(step_input, [state])
            state = next_state[0]
            scores = tf.matmul(tf.expand_dims(state, 1), enc_out_transformed, transpose_b=True)
            attn_weights = tf.nn.softmax(scores, axis=-1)
            context_vector = tf.matmul(attn_weights, enc_out)
            context_vector = tf.squeeze(context_vector, axis=1)
            combined = tf.concat([state, context_vector], axis=-1)
            logits = self.dense(combined)
            all_logits.append(tf.expand_dims(logits, axis=1))
        return tf.concat(all_logits, axis=1)
    
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids)
        enc_out, enc_state = self.encoder(enc_emb)
        return enc_out, [enc_state]
    
    def get_next_token(self, x, state, enc_out):
        step_input = self.embed_layer(x)
        _, next_state = self.decoder_cell(step_input, state)
        curr_state = next_state[0]
        enc_out_transformed = self.dense_attn(enc_out)
        scores = tf.matmul(tf.expand_dims(curr_state, 1), enc_out_transformed, transpose_b=True)
        attn_weights = tf.nn.softmax(scores, axis=-1)
        context_vector = tf.matmul(attn_weights, enc_out)
        context_vector = tf.squeeze(context_vector, axis=1)
        combined = tf.concat([curr_state, context_vector], axis=-1)
        logits = self.dense(combined)
        out = tf.argmax(logits, axis=-1, output_type=tf.int32)
        return out, next_state

# Loss函数以及训练逻辑

In [4]:
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(4000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [5]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.298468
step 500 : loss 1.2429826
step 1000 : loss 0.3366822
step 1500 : loss 0.121608235
step 2000 : loss 0.061163377
step 2500 : loss 0.036551956
step 3000 : loss 0.01764362
step 3500 : loss 0.015390238


<tf.Tensor: shape=(), dtype=float32, numpy=0.01702527329325676>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [6]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
[('YUSOPVJERNELEZKWYBDA', 'ADBYWKZELENREJVPOSUY'), ('VPWHWQQFBAKRFKKSSLVW', 'WVLSSKKFRKABFQQWHWPV'), ('JXQAWYWIJUVOVBEYLPYY', 'YYPLYEBVOVUJIWYWAQXJ'), ('CAFIORIMALUSJJNLPJLL', 'LLJPLNJJSULAMIROIFAC'), ('RTXFPMJWSRCEBSCMRYMG', 'GMYRMCSBECRSWJMPFXTR'), ('PTOJKUYXYBEHQCXRHRTX', 'XTRHRXCQHEBYXYUKJOTP'), ('SRGQAYFKMPZMRTBNWOQR', 'RQOWNBTRMZPMKFYAQGRS'), ('ZIUWZDFFWOMFHESAJAKN', 'NKAJASEHFMOWFFDZWUIZ'), ('QKJTHOSNYISDZLPFDRPX', 'XPRDFPLZDSIYNSOHTJKQ'), ('LRZQYVNFGAYRCWWJGBNV', 'VNBGJWWCRYAGFNVYQZRL'), ('TMYIOQEQBHJFHYTEUIUP', 'PUIUETYHFJHBQEQOIYMT'), ('QUFTTDZGTYRTWBLPUPEQ', 'QEPUPLBWTRYTGZDTTFUQ'), ('NPAGSIDSMRWFMLYRVIML', 'LMIVRYLMFWRMSDISGAPN'), ('GZRYNTMHMCAHHKXNAYKJ', 'JKYANXKHHACMHMTNYRZG'), ('HHQPRQOXYUVXRZQDNVYQ', 'QYVNDQZRXVUYXOQRPQHH'), ('HVIEKPYIMDIHUPHCIUUK', 'KUUICHPUHIDMIYPKEIVH'), ('VPON